In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import random
from datetime import datetime, timedelta

In [0]:
df = spark.read.table("my_catalog.bsheffield_spark.nhs_combined_dataset").toPandas()

In [0]:
def datetime_change(df):
    df = df.copy()
    df['issue_date'] = pd.to_datetime(df['issue_date'], errors='coerce')
    df['closing_date'] = pd.to_datetime(df['closing_date'], errors='coerce')
    df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
    return df

def bin_delivery_target(row):
    return 1 if row['quoted_lead_days'] == row['actual_lead_days'] else 0

def assign_month(df):
    df = df.copy()
    df['closing_date_month'] = df['closing_date'].dt.month_name()
    return df

In [0]:
df=datetime_change(df)
df['delivery_target'] = df.apply(bin_delivery_target, axis=1)
df = assign_month(df)


In [0]:
df_0_time = df.query("delivery_target == 0")

In [0]:
df_0_time['percentage_diff_qt_vs_at'] = (df_0_time['actual_lead_days'] - df_0_time['quoted_lead_days']) / df_0_time['quoted_lead_days']


/home/spark-6165127b-dfca-4d89-84b2-30/.ipykernel/2767/command-6018119446537361-4165038632:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_0_time['percentage_diff_qt_vs_at'] = (df_0_time['actual_lead_days'] - df_0_time['quoted_lead_days']) / df_0_time['quoted_lead_days']


In [0]:
month_stats = (
    df_0_time
    .groupby(['closing_date_month', 'contract_type'])['percentage_diff_qt_vs_at']
    .quantile([0.20,0.30, 0.40,0.50, 0.60,0.70, 0.80, 0.95])
    .unstack()
)
month_stats_spark = spark.createDataFrame(month_stats.reset_index())
month_stats_spark.write.mode("overwrite").saveAsTable("my_catalog.bsheffield_spark.time_month_stats")

In [0]:
def percent_increase_prediction_time(row, month_stats):
    prob = row['Predicted_Probability']
    quoted_days = row['quoted_lead_days']
    month = row['closing_date_month']
    ctype = row['contract_type']

    # If month or contract type missing, fall back to global median
    if (month, ctype) not in month_stats.index:
        fallback = month_stats.mean().mean()  # global median uplift
        return quoted_days * (1 + fallback)

    # Extract quantiles safely
    qvals = month_stats.loc[(month, ctype)]

    q20 = qvals.get(0.20, qvals.median())
    q30 = qvals.get(0.30, qvals.median())
    q40 = qvals.get(0.40, qvals.median())
    q50 = qvals.get(0.50, qvals.median())
    q60 = qvals.get(0.60, qvals.median())
    q70 = qvals.get(0.70, qvals.median())
    q80 = qvals.get(0.80, qvals.median())
    q95 = qvals.get(0.95, qvals.median())

    # Map probability → quantile
    if prob <= 0.2:
        uplift = q95
    elif prob <= 0.3:
        uplift = q80
    elif prob <= 0.4:
        uplift = q70
    elif prob <= 0.5:
        uplift = q60
    elif prob <= 0.6:
        uplift = q50
    elif prob <= 0.7:
        uplift = q40
    elif prob <= 0.8:
        uplift = q30
    else:
        uplift = q20

    # Final predicted maximum lead time
    return quoted_days * (1 + uplift)


In [0]:
TIME_lin_class_results_df = spark.read.table("my_catalog.bsheffield_spark.TIME_lin_class_results_df").toPandas()

In [0]:
time_preds = TIME_lin_class_results_df.copy()
time_preds['predicted_max_lead_time'] = time_preds.apply(percent_increase_prediction_time, axis=1, args=(month_stats,))

In [0]:
time_preds['predicted_max_lead_time'] = time_preds['predicted_max_lead_time'].astype(int)

In [0]:
(
    spark.createDataFrame(time_preds)
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("my_catalog.bsheffield_spark.TIME_lin_class_results_df")
)
